# 03 - Hybrid retrieval and ranking

The first stage finds a small group of possible listings. The second stage scores that group in more detail. This keeps the search fast and makes the final order easier to explain.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from app.data import load_interactions, load_properties
from app.features.user_features import user_preferences_from_history
from app.pipelines.train_ranker import build_training_data
from app.ranking.baseline_ranker import BaselineRanker, RANKING_FEATURES
from app.ranking.diversity_reranker import diversify
from app.ranking.lgbm_ranker import LearnedRanker
from app.recommendation.explanations import recommendation_reasons
from app.recommendation.recommender import PropertyRecommender
from app.retrieval.hybrid_retriever import HybridRetriever

properties = load_properties()
interactions = load_interactions()

## Select one user

An active user is selected so both content and review-history signals can be checked.

In [ ]:
user_counts = interactions['user_id'].value_counts()
user_id = str(user_counts.index[0])
history_ids = interactions.loc[interactions['user_id'].astype(str) == user_id, 'property_id'].astype(str)
history = properties[properties['property_id'].astype(str).isin(history_ids)]
preferences = user_preferences_from_history(properties, interactions, user_id)
preferences['query'] = 'Berlin entire apartment Wifi kitchen'

print('User:', user_id)
print('History size:', len(history))
print('Preferences:', preferences)
display(history[['property_id', 'title', 'neighborhood', 'price', 'amenities']])

## Stage 1: candidate retrieval

Hard filters remove unavailable or unsuitable listings. TF-IDF search and item co-occurrence then create the candidate set. Seen listings are removed for this user.

In [ ]:
retriever = HybridRetriever(properties, interactions)
candidate_count = min(100, len(properties))
candidates = retriever.retrieve(
    preferences,
    user_id=user_id,
    candidate_count=candidate_count,
    exclude_seen=True,
)

retrieval_columns = [
    'property_id', 'title', 'price', 'semantic_score',
    'collaborative_score', 'retrieval_score',
]
print(f'Catalog size: {len(properties):,}')
print(f'Candidates after retrieval: {len(candidates):,}')
display(candidates[retrieval_columns].head(15).round(3))

In [ ]:
if not candidates.empty:
    candidates.set_index('property_id')[['semantic_score', 'collaborative_score']].head(15).plot(
        kind='bar', figsize=(11, 4), title='Retrieval signals'
    )
    plt.ylabel('Score')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Stage 2: weighted ranking

The baseline ranker adds budget, amenity, quality, popularity and availability scores. The weights are fixed, so this model works without training.

In [ ]:
baseline_ranker = BaselineRanker()
ranked = baseline_ranker.rank(candidates, preferences)
ranking_columns = ['property_id', 'title', *RANKING_FEATURES, 'ranking_score']
display(ranked[ranking_columns].head(15).round(3))

In [ ]:
weights = pd.Series({
    'semantic_score': 0.32,
    'collaborative_score': 0.18,
    'budget_score': 0.18,
    'amenity_score': 0.12,
    'quality_score': 0.12,
    'popularity_score': 0.05,
    'availability_score': 0.03,
}).sort_values()
weights.plot(kind='barh', figsize=(8, 4), color='#047857', title='Weighted ranker settings')
plt.xlabel('Weight')
plt.tight_layout()
plt.show()

## Diversity and recommendation reasons

The final list keeps at most two properties from one neighbourhood. This avoids showing many near-identical results.

In [ ]:
top_k = min(10, len(ranked))
final_ranked = diversify(ranked, top_k=top_k, max_per_neighborhood=2)
final_ranked['reasons'] = [
    recommendation_reasons(row, preferences)
    for row in final_ranked.to_dict('records')
]
display(final_ranked[['property_id', 'title', 'neighborhood', 'ranking_score', 'reasons']].round(3))

## Train the LightGBM ranker

Each user is one ranking group. Reviewed listings are positive labels. Other retrieved listings are negative examples.

In [ ]:
training_frame, labels, groups = build_training_data(
    max_users=min(300, interactions['user_id'].nunique()),
    candidate_count=min(40, len(properties)),
)
print(f'Training rows: {len(training_frame):,}')
print(f'User groups: {len(groups):,}')
print(f'Positive rate: {sum(labels) / len(labels):.3f}')
display(training_frame[RANKING_FEATURES].describe().round(3))

In [ ]:
learned_ranker = LearnedRanker()
learned_ranker.fit(training_frame, labels, groups)
training_frame = training_frame.copy()
training_frame['learned_score'] = learned_ranker.predict(training_frame)

importance = pd.Series(
    learned_ranker.model.feature_importances_,
    index=RANKING_FEATURES,
    name='importance',
).sort_values()
display(importance.sort_values(ascending=False).to_frame())
if importance.sum() == 0:
    print('The sample data is too small for useful feature importance. Run this step with the real data.')
importance.plot(kind='barh', figsize=(8, 4), color='#2563eb', title='LightGBM feature importance')
plt.tight_layout()
plt.show()

## Compare served results

This uses the same recommender class as the FastAPI endpoint.

In [ ]:
baseline_service = PropertyRecommender(properties, interactions)
learned_service = PropertyRecommender(properties, interactions, learned_ranker=learned_ranker)

baseline_results = baseline_service.recommend(preferences, user_id=user_id, top_k=5)
learned_results = learned_service.recommend(preferences, user_id=user_id, top_k=5)

comparison = pd.DataFrame({
    'weighted_baseline': [item['id'] for item in baseline_results],
    'lightgbm_ranker': [item['id'] for item in learned_results],
})
display(comparison)

## Notes

- Retrieval should return enough relevant candidates before ranking is tuned.
- Fixed weights are easy to explain and make a strong first baseline.
- LightGBM should only be used in the API after it wins on time-based evaluation.
- Feature importance helps with debugging, but it does not prove that one feature causes a result.